# Benchmark run analysis

Comparing benchmark runs

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
RUN_A = 'store/phlag/gaussian/w5k_s5k/rho0.9_beta4.0'
RUN_B = 'store/phlag/gaussian/w5k_s5k/ilr/rho0.9_beta4.0'

run_dir_a = Path(RUN_A)
run_dir_b = Path(RUN_B)

runs_path_a = run_dir_a / "runs.tsv"
runs_path_b = run_dir_b / "runs.tsv"

df_a = pd.read_csv(runs_path_a, sep="\t")
df_b = pd.read_csv(runs_path_b, sep="\t")

df_b.columns

In [ ]:
print(df_a['clip_activation_count'].mean())
print(df_b['clip_activation_count'].mean())

In [ ]:
df_a[df_a['em_gt_hd'] < 0.4][['run_id', 'em_gt_hd']]

In [ ]:
df_a[df_a['f1']==1]#.value_counts()

In [ ]:
df_a[df_a['alt_prop_pred']==pd.NA]

In [ ]:
PATH_COLUMNS = ["report_path"]
PANEL_COLUMNS = ["in_panel_a", "in_panel_b", "in_panel_relerr", "in_panel_em_divergence", "anomaly_fraction_source"]
ID_COLUMNS = ["run_id", "source_leaf", 'x_variable']
DROP_COLUMNS = PATH_COLUMNS + PANEL_COLUMNS + ID_COLUMNS

before_cols = set(df_a.columns)
df_a = df_a.drop(columns=DROP_COLUMNS, errors="ignore")
df_b = df_b.drop(columns=DROP_COLUMNS, errors="ignore")
after_cols = set(df_a.columns)

print("Deleted:", sorted(before_cols - after_cols))
print("Added:", sorted(after_cols - before_cols))

In [ ]:
total = df_b['n_windows'][0]
alt_a = df_a['fp'] + df_a['tp']
alt_b = df_b['fp'] + df_b['tp']
print(np.mean(alt_a / total), np.mean(alt_b / total))

In [ ]:
df_a['pooled_mean_norm']

In [ ]:
(df_b['pooled_mean_norm'] - df_a['pooled_mean_norm']).unique()

In [ ]:
len(df_a[(df_a['transition_null_to_null']==1.0) | (df_a['transition_alt_to_alt']==1.0)])

In [ ]:
df_a[df_a['em_hd'] < df_a['em_gt_hd']]#[['em_hd', 'em_gt_hd']]

In [ ]:
a = df_a.groupby(['x_bin'])['f1']
b = df_b.groupby(['x_bin'])['f1']
f1_branch_change_mean = (b.mean() - a.mean()) / a.mean()
print('F1 Mean Relative Change Over Branches\n', f1_branch_change_mean)
f1_branch_change_std = (b.std() - a.std()) / a.std()
print('F1 STD Relative Change Over Branches\n', f1_branch_change_std)
# STD shouldn't be affected by mean changing unless prvious STD was bounded at 0 or 1 (since F1 is bounded by nature)

In [ ]:
f1_mean_change = (
    df_b.groupby(['category', 'subcategory'])['f1'].mean()
    - df_a.groupby(['category', 'subcategory'])['f1'].mean()
) / df_a.groupby(['category', 'subcategory'])['f1'].mean()
f1_std_change = (
    df_b.groupby(['category', 'subcategory'])['f1'].std()
    - df_a.groupby(['category', 'subcategory'])['f1'].std()
) / df_a.groupby(['category', 'subcategory'])['f1'].std()
print('F1 Relative Mean Change Over Changes\n', f1_mean_change)
print('F1 Relative STD Change Over Changes\n', f1_std_change)
# print(df_c.groupby(['category', 'subcategory'])['em_bd'].mean())

In [ ]:
print(np.mean(df_a[['fp', 'tn', 'fn', 'tp']], axis=0) / np.sum(np.mean(df_a[['fp', 'tn', 'fn', 'tp']], axis=0)))
print(np.mean(df_b[['fp', 'tn', 'fn', 'tp']], axis=0) / np.sum(np.mean(df_b[['fp', 'tn', 'fn', 'tp']], axis=0)))

In [ ]:
filter = 'x_bin'
target = 'f1'
value1 = "(0,0.1]"
value2 = "(0.1,0.25]"
low_drop = np.mean(df_b[df_b[filter]==value1][target] - df_a[df_a[filter]==value1][target])
high_drop = np.mean(df_b[df_b[filter]==value2][target] - df_a[df_a[filter]==value2][target])
print(low_drop, high_drop)